# NB18 — Auditoría simétrica y trazabilidad de representaciones (AgriEngineering 4664292)
**Manuscrito:** *Geometric Invariance and Representation Sufficiency in Morphometric Bean and Rice Classification*

**Problema editorial:** Revisor 2, comentarios 2, 3, 5 y 7; control complementario de Revisor 1 comentario 1. Se evalúan de manera paralela los contrastes entre características nativas (R1), núcleo medido (R2), base invariante de cinco descriptores y base invariante + log-área en Dry Bean e INIAP Rice. Dry Bean contiene también su conjunto invariante completo de ocho descriptores.

**Importante:** esta es una **auditoría adicional post revisión** sobre predicciones congeladas del NB12; **no** reentrena, no optimiza hiperparámetros, no cambia splits, **no** da equivalencia agronómica demostrada y **no** ejecuta una prueba de transferencia entre cultivos. CPU es suficiente.

**Fuentes:** ZIP NB12 en Mi unidad/Grain_Robustness_Q1 (SHA-256 verificado); test y calibración congelados. **Salida:** CSV, figuras, metadatos, hashes y ZIP único tanto en My Drive como descargable al finalizar.

## 1. Dependencias, configuración y Drive
No cargamos imágenes originales INIAP. Instalación automática de `pyarrow` únicamente si falta en Colab.

In [ ]:
import sys, subprocess, importlib.util
needed = [('pyarrow','pyarrow'),('pandas','pandas'),('scikit-learn','sklearn'),('matplotlib','matplotlib')]
missing=[p for p,m in needed if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import json, io, zipfile, hashlib, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score
from google.colab import drive, files as colab_files
from IPython.display import display
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/Grain_Robustness_Q1')
REVIEW=PROJECT/'Revision_4664292_Ensayos_2026'
OUT=REVIEW/'resultados_colab'/'NB18_4664292'
if not PROJECT.is_dir(): raise FileNotFoundError(f'Carpeta proyecto no encontrada: {PROJECT}')
REVIEW.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
SOURCE=PROJECT/'NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip'
if not SOURCE.is_file():
    candidates=list(PROJECT.rglob(SOURCE.name))
    if len(candidates)!=1: raise FileNotFoundError(f'ZIP NB12: encontrados {len(candidates)} candidatos: {candidates}')
    SOURCE=candidates[0]
EXPECTED_SHA='68eee4090933b5fb31e611f8175c773ee3395af979e3729db68af61b06da05fa'
SOURCE_SHA=hashlib.file_digest(SOURCE.open('rb'),'sha256').hexdigest()
assert SOURCE_SHA==EXPECTED_SHA, f'¡Fuente NB12 inesperada! {SOURCE_SHA}'
SEED=20261027
N_BOOT=2000
MARGINS=(0.005,0.010,0.020)
print('✅ NB12 verificado:',SOURCE,'\nSHA-256:',SOURCE_SHA)
print('✅ Resultados:',OUT,'| CPU | remuestreos:',N_BOOT)

## 2. Lectura congelada: todas las representaciones y su separación entre calibración y test
Se evalúan **las diez representaciones archivadas**, no solo el modelo seleccionado. El algoritmo comprueba predicciones alineadas por `record_id`, verdad y grupo. `PASS` de esta sección no demuestra ausencia de fugas desde entrenamiento; el control específico de test/calibración complementa NB17B.

In [ ]:
REPS={
'dry_bean':{'R1':'R1_full_native','R2':'R2_measured_core','R3-C':'R3_complete_invariant',
            'R3-5':'R3_reduced_current','R4-C':'R4_complete_log_area','R4-5':'R4_reduced_log_area_current'},
'iniap_rice':{'R1':'R1_full_native','R2':'R2_measured_core',
             'R3-5':'R3_complete_invariant','R4-5':'R4_complete_log_area'},
}
CONTRASTS={
'dry_bean':[('R1','R2'),('R1','R3-C'),('R1','R3-5'),('R1','R4-C'),('R1','R4-5'),
            ('R3-C','R3-5'),('R4-C','R4-5')],
'iniap_rice':[('R1','R2'),('R1','R3-5'),('R1','R4-5')],
}
BASE='06_predictions'
MODEL_TABLE='09_tables/12_REVIEWER_REANALYSIS/Table_05_corrected_representation_test_metrics.csv'
AUDIT_TABLE='09_tables/12_REVIEWER_REANALYSIS/Table_01_representation_audit.csv'
PRED={};SEP=[];MET=[]
with zipfile.ZipFile(SOURCE) as z:
    assert z.testzip() is None,'ZIP corrupto'
    inv=z.namelist()
    model_ref=pd.read_csv(io.BytesIO(z.read(MODEL_TABLE)))
    representation_ref=pd.read_csv(io.BytesIO(z.read(AUDIT_TABLE)))
    for ds,mapping in REPS.items():
        reference=None
        for rep,internal in mapping.items():
            frames={}
            for part in ['calibration','test']:
                path=f'{BASE}/{ds}/{internal}/rbf_svm_{part}_predictions.parquet'
                if path not in inv: raise FileNotFoundError(path)
                data=pd.read_parquet(io.BytesIO(z.read(path)))
                cols={'record_id','duplicate_group_id','class_label','predicted_native'}
                if not cols.issubset(data): raise ValueError(f'Faltan columnas {cols-set(data)}: {path}')
                for c in cols:data[c]=data[c].astype(str)
                if not data.record_id.is_unique:raise AssertionError(f'IDs repetidos: {path}')
                if data.groupby('duplicate_group_id').class_label.nunique().max()!=1:
                    raise AssertionError(f'Grupo multiclase: {path}')
                frames[part]=data.sort_values('record_id').reset_index(drop=True)
            ca,te=frames['calibration'],frames['test']
            overlap_ids=len(set(ca.record_id)&set(te.record_id))
            overlap_groups=len(set(ca.duplicate_group_id)&set(te.duplicate_group_id))
            SEP.append(dict(dataset=ds,representation=rep,n_cal=len(ca),cal_groups=ca.duplicate_group_id.nunique(),
                            n_test=len(te),test_groups=te.duplicate_group_id.nunique(),
                            overlap_record_id=overlap_ids,overlap_duplicate_group_id=overlap_groups,
                            verdict='PASS' if overlap_ids==overlap_groups==0 else 'FAIL'))
            key=te[['record_id','duplicate_group_id','class_label']]
            if reference is None:reference=key
            elif not key.equals(reference): raise AssertionError(f'Test desalineado en {ds}/{rep}')
            got=f1_score(te.class_label,te.predicted_native,average='macro')
            ref=model_ref[(model_ref.dataset_id==ds)&(model_ref.representation==internal)]
            if len(ref)!=1 or not np.isclose(got,float(ref.iloc[0].macro_f1),atol=1e-10):
                raise AssertionError(f'Macro-F1 no coincide NB12: {ds}/{rep} {got}')
            PRED[(ds,rep)]=te
            row=representation_ref[(representation_ref.dataset_id==ds)&(representation_ref.representation==internal)].iloc[0]
            MET.append(dict(dataset=ds,representation=rep,nb12_internal_name=internal,n_features=int(row.n_features),
                            features=str(row.features),n_test=len(te),macro_f1=got,
                            accuracy=accuracy_score(te.class_label,te.predicted_native)))
sep=pd.DataFrame(SEP);metrics=pd.DataFrame(MET)
sep.to_csv(OUT/'NB18_01_independencia_por_representacion.csv',index=False)
metrics.to_csv(OUT/'NB18_02_rendimiento_y_esquema.csv',index=False)
display(sep);display(metrics[['dataset','representation','n_features','macro_f1','accuracy']])
if (sep.verdict!='PASS').any(): raise AssertionError('NO INTEGRAR: existe solapamiento calibración/test')
print('✅ 10 representaciones verificadas y alineadas con NB12; test/calibración disjuntos.')

## 3. Definiciones simétricas y límites de comparabilidad
Los 5 descriptores adimensionales de R3-5 son `aspect_ratio, extent, solidity, circularity, ellipse_fill`; R4-5 incorpora `log_area`. En Dry Bean también mostramos R3-C completo (8 dimensiones) y R4-C completo+área (9). **La etiqueta R3-C jamás debe equipararse a R3-5.** El contraste R1–R3-5 no iguala el número de variables originales entre cultivos, ni prueba transferencia causal de escala.

In [ ]:
BASE5=['aspect_ratio','extent','solidity','circularity','ellipse_fill']
for ds in REPS:
    found=metrics[(metrics.dataset==ds)&(metrics.representation=='R3-5')].iloc[0]
    actual=[x.strip() for x in found.features.split('|')]
    assert actual==BASE5,f'Base cinco descriptores inconsistente: {ds} {actual}'
    logrow=metrics[(metrics.dataset==ds)&(metrics.representation=='R4-5')].iloc[0]
    actual_log=[x.strip() for x in logrow.features.split('|')]
    assert actual_log==BASE5+['log_area'],f'R4-5 inconsistentes: {ds}: {actual_log}'
print('✅ Ambas bases R3-5 y R4-5 son análogas por nombre y definición operacional.')
print('⚠️ No se afirma que sea idéntico el procedimiento de adquisición de imágenes, lotes ni las clases.')

## 4. Bootstrap agrupado pareado; sensibilidad de márgenes exploratorios
El remuestreo es por `duplicate_group_id` y estratificado por clase; comparación sobre el **mismo test** en cada dataset. IC percentil 90 % y 95 %, 2.000 remuestreos. Las etiquetas *inside ± margin* son **análisis exploratorios post hoc**, no prueba automática de equivalencia agronómica. **No convertir ausencia de significación en equivalencia.**

In [ ]:
def paired_bootstrap(ds,A,B,n_boot=N_BOOT):
    a=PRED[(ds,A)]
    b=PRED[(ds,B)]
    assert a[['record_id','duplicate_group_id','class_label']].equals(b[['record_id','duplicate_group_id','class_label']])
    y=a.class_label.to_numpy(); pa=a.predicted_native.to_numpy();pb=b.predicted_native.to_numpy()
    labs=sorted(a.class_label.unique())
    # Una misma agrupación no puede atravesar clases
    groups=[]
    for cl in labs:
        mask=np.where(y==cl)[0]
        cls_groups=a.iloc[mask].groupby('duplicate_group_id',sort=True).indices
        groups.append([mask[idx] for idx in cls_groups.values()])
    point=f1_score(y,pa,labels=labs,average='macro')-f1_score(y,pb,labels=labs,average='macro')
    # Semilla diferente y estable por dataset y contraste, sin hash() aleatorizado de Python
    seed_shift=sum((i+1)*ord(ch) for i,ch in enumerate(ds+'_'+A+'_'+B))
    rng=np.random.default_rng(SEED+seed_shift)
    diffs=np.empty(n_boot)
    for k in range(n_boot):
        chosen=[]
        for stratum in groups:
            picks=rng.integers(0,len(stratum),size=len(stratum))
            chosen.extend(stratum[t] for t in picks)
        inds=np.concatenate(chosen)
        diffs[k]=f1_score(y[inds],pa[inds],labels=labs,average='macro',zero_division=0)-f1_score(y[inds],pb[inds],labels=labs,average='macro',zero_division=0)
    ci90=np.quantile(diffs,[.05,.95]); ci95=np.quantile(diffs,[.025,.975])
    obj=dict(dataset=ds,comparison=f'{A} minus {B}',delta_macro_f1=point,
             ci90_low=ci90[0],ci90_high=ci90[1],ci95_low=ci95[0],ci95_high=ci95[1],
             n_test=len(a),n_duplicate_groups=a.duplicate_group_id.nunique(),n_boot=n_boot,
             test_selection='predeclared_review_sensitivity')
    for m in MARGINS:
        label=f'ci90_inside_margin_{str(m).replace(".","p")}_exploratory'
        obj[label]=bool(ci90[0]>-m and ci90[1]<m)
    return obj
comparisons=[]
for ds,pairs in CONTRASTS.items():
    for A,B in pairs:
        obj=paired_bootstrap(ds,A,B)
        comparisons.append(obj)
        print(ds,obj['comparison'],f"Delta={obj['delta_macro_f1']:.5f}",
              f"IC90=[{obj['ci90_low']:.5f},{obj['ci90_high']:.5f}]")
contrasts=pd.DataFrame(comparisons)
contrasts.to_csv(OUT/'NB18_03_contrastes_pareados_grupales.csv',index=False)
display(contrasts)

## 5. F1 por clase, apoyo descriptivo y figuras de contraste
Se incluyen todas las clases de ambas bases con el mismo test congelado. Los gráficos son descriptivos; mostrar diferencias entre cultivos no identifica un efecto causal de origen.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
class_rows=[]
for (ds,rep),d in PRED.items():
    labs=sorted(d.class_label.unique())
    P,R,F,S=precision_recall_fscore_support(d.class_label,d.predicted_native,labels=labs,zero_division=0)
    for lab,pv,rv,fv,n in zip(labs,P,R,F,S):
        class_rows.append(dict(dataset=ds,representation=rep,class_label=lab,n_test=int(n),precision=pv,recall=rv,f1=fv))
pd.DataFrame(class_rows).to_csv(OUT/'NB18_04_f1_por_clase.csv',index=False)
# Solo familias alineadas R1/R2/R3-5/R4-5, no mezclar R3-C y R3-5 bajo el mismo nombre
select=metrics[metrics.representation.isin(['R1','R2','R3-5','R4-5'])]
fig,ax=plt.subplots(figsize=(10,5.7))
labels=['R1','R2','R3-5','R4-5'];x=np.arange(len(labels));w=.35
for i,ds in enumerate(['dry_bean','iniap_rice']):
    vals=[float(select[(select.dataset==ds)&(select.representation==r)].macro_f1.iloc[0]) for r in labels]
    bars=ax.bar(x+(i-.5)*w,vals,w,label='Dry Bean' if i==0 else 'INIAP Rice')
    for bar,val in zip(bars,vals): ax.annotate(f'{val:.3f}',(bar.get_x()+bar.get_width()/2,val),ha='center',xytext=(0,3),textcoords='offset points',fontsize=9)
ax.set_xticks(x,labels);ax.set_ylim(0,1.08);ax.set_ylabel('Test macro-F1');ax.set_title('Matched five-descriptor representation roles: frozen test');ax.legend();ax.grid(axis='y',alpha=.15)
fig.tight_layout();fig.savefig(OUT/'NB18_Fig_01_comparacion_bases_cinco.png',dpi=210);plt.close(fig)
print('✅ F1 por clase y figura guardados en:',OUT)

## 6. Integridad, metadatos y ZIP único descargable
**La salida no sustituye NB12–NB17B.** Conserva un registro propio para enviar en el chat, depositar junto al notebook en Drive y anexar eventualmente al repositorio. La generación NO modifica archivos fuente.

In [ ]:
metadata=dict(manuscript='agriengineering-4664292',notebook='NB18_Representaciones_Simetricas_4664292_CPU',
              run_utc=datetime.now(timezone.utc).isoformat(),source=SOURCE.name,source_nb12_sha256=SOURCE_SHA,
              seed=SEED,bootstraps=N_BOOT,comparisons=CONTRASTS,margin_sensitivity=list(MARGINS),
              datasets=list(REPS), all_independent=bool((sep.verdict=='PASS').all()),
              python=platform.python_version(),pandas=pd.__version__,numpy=np.__version__,
              limitations=['Post-review exploratory sensitivity, not prespecified hypothesis',
                'Within-dataset fitted models only; no transfer across datasets',
                'Group bootstrap CI percentiles do not agronomically justify a margin',
                'No verification of all upstream training leakage or imaging provenance'])
(OUT/'NB18_05_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT/'NB18_06_LEEME.txt').write_text('AgriEngineering 4664292 — NB18\n'
    'Diez modelos del NB12 congelado, contraste de bases cinco y completo, independencia cal/test por representación.\n'
    'Sin reentrenamiento. Mantener la etiqueta exploratoria y no inferir validez en lotes futuros.\n',encoding='utf-8')
files_to_hash=sorted(x for x in OUT.glob('NB18_*') if x.is_file() and x.name!='NB18_07_SHA256.csv')
hashes=pd.DataFrame([dict(file=x.name,bytes=x.stat().st_size,sha256=hashlib.sha256(x.read_bytes()).hexdigest()) for x in files_to_hash])
hashes.to_csv(OUT/'NB18_07_SHA256.csv',index=False)
zip_path=REVIEW/'NB18_REPRESENTACIONES_SIMETRICAS_4664292.zip'
outputs=sorted(x for x in OUT.glob('NB18_*') if x.is_file())
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for x in outputs:z.write(x,arcname=f'NB18_4664292/{x.name}')
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
    assert len(z.namelist())==len(outputs)
print('✅ ZIP guardado en My Drive:',zip_path)
print('✅ Contenido:',len(outputs),'archivos')
print('✅ ZIP SHA-256:',hashlib.sha256(zip_path.read_bytes()).hexdigest())
print('📎 Envíame este ZIP por el chat para auditar NB18.')
colab_files.download(str(zip_path))